In [1]:
from political_argumentation_rag.datatypes.dataclasses import KnowledgeBaseConfig

In [2]:
from political_argumentation_rag.graph_rag import GraphBasedRetrieval
from political_argumentation_rag.datatypes.dataclasses import PoliticalFilter, UserDefinedExamplesConfig, TypicalResponsesConfig, GraphRAGConfig, Utterance, UtteranceType, VectorBasedConfig
from political_argumentation_rag.datatypes.enums import PoliticalPositionEnsembleOrModelName, IllocutionaryForce, QueryDirection, RelationType

d:\Post-Doc\Validating-Political-Position-Predictions-in-Arguments\examples\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# KB configuration settings -- defaults to credentials in /docker/Dockerfile.neo4j
kb_config = KnowledgeBaseConfig()

In [4]:
# political filter set to left-wing position (i.e. 10-30)

political_filter = PoliticalFilter(
    PoliticalPositionEnsembleOrModelName.ENSEMBLE_1_ALL_MODELS,
    position_min=10,
    position_max=30,
    position_std=10,
    probability_of_na=0.05
)

config = GraphRAGConfig(political_filter)

In [5]:
utt = Utterance("The covid vaccine is safe", locution_or_proposition=UtteranceType.LOCUTION, illocutinary_force=IllocutionaryForce.ASSERTING)

In [6]:
graph_config = GraphBasedRetrieval(kb=kb_config,config=config,)

d:\Post-Doc\Validating-Political-Position-Predictions-in-Arguments\examples\.venv\Lib\site-packages\political_argumentation_rag\graph_rag.py:33: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  self.kb_connection = Neo4jGraph(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1316.46it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Hybrid Graph-based Retrieval

In this approach, we use cosine similarity between the input utterance and all nodes in the graph and then use the relations from each node to find pertinent examples of, say, conflict, rephrase, chronological transition, etc

## Testing retrieval of enumerated examples

In [7]:
typical_replies_config = TypicalResponsesConfig(3, QueryDirection.BOTH)

relevant_examples = graph_config.get_typical_responses(utt, typical_replies_config)

relevant_examples

[RetrievedExample(retrieval_method='hybird-vector-and-graph-based', content_type='locution', relation='SUPPORTS', input_text='The covid vaccine is safe', similar_illocutionary_force='Asserting', related_illocutionary_force='Asserting', similar_text='What we are calling on the UK Government to do is to ease the intellectual property rules around who owns this vaccine technology, who owns the vaccine recipes', related_text='making the pie bigger', example_text_tuple=('making the pie bigger', 'What we are calling on the UK Government to do is to ease the intellectual property rules around who owns this vaccine technology, who owns the vaccine recipes'), query_direction='cosim(input, similar_node)<-[relation]-(example)'),
 RetrievedExample(retrieval_method='hybird-vector-and-graph-based', content_type='locution', relation='SUPPORTS', input_text='The covid vaccine is safe', similar_illocutionary_force='Asserting', related_illocutionary_force='Asserting', similar_text='what we are calling on

## Testing retrieval of specific, user-chosen examples

In [8]:
relation_choices = [RelationType.SUPPORT, RelationType.CONFLICT]
query_node_choices = [IllocutionaryForce.ASSERTING, IllocutionaryForce.CHALLENGING]

user_defined_config = UserDefinedExamplesConfig(
    relation_choices=relation_choices,
    query_node_choices=query_node_choices,
    num_examples=2,
    query_direction=QueryDirection.BOTH
)

user_defined_examples = graph_config.get_explicit_examples(utt, example_config=user_defined_config)
user_defined_examples

[RetrievedExample(retrieval_method='hybird-vector-and-graph-based', content_type='locution', relation='SUPPORTS', input_text='The covid vaccine is safe', similar_illocutionary_force='Asserting', related_illocutionary_force='Asserting', similar_text='What we are calling on the UK Government to do is to ease the intellectual property rules around who owns this vaccine technology, who owns the vaccine recipes', related_text='making the pie bigger', example_text_tuple=('making the pie bigger', 'What we are calling on the UK Government to do is to ease the intellectual property rules around who owns this vaccine technology, who owns the vaccine recipes'), query_direction='cosim(input, similar_node)<-[relation]-(example)'),
 RetrievedExample(retrieval_method='hybird-vector-and-graph-based', content_type='locution', relation='SUPPORTS', input_text='The covid vaccine is safe', similar_illocutionary_force='Asserting', related_illocutionary_force='Asserting', similar_text='what we are calling on

# Vector-based Retrieval

This approach simply finds the N most similar nodes to the input text. You can choose what illocutionary force you want the retrieve nodes to have.

In [9]:
vector_based_config = VectorBasedConfig(
    num_examples=5,
    similar_nodes_illocutionary_force=IllocutionaryForce.ASSERTIVE_QUESTIONING
)

user_defined_examples = graph_config.get_similar_examples(utt, vector_based_config)
user_defined_examples

[RetrievedExample(retrieval_method='vector-based', content_type='locution', relation='', input_text='The covid vaccine is safe', similar_illocutionary_force='Assertive Questioning', related_illocutionary_force='', similar_text=" what you can't be is selfish and keep it all for yourself, you know", related_text='', example_text_tuple=(), query_direction=''),
 RetrievedExample(retrieval_method='vector-based', content_type='proposition', relation='', input_text='The covid vaccine is safe', similar_illocutionary_force='Assertive Questioning', related_illocutionary_force='', similar_text="you can't be selfish and keep the vaccines all for yourself", related_text='', example_text_tuple=(), query_direction=''),
 RetrievedExample(retrieval_method='vector-based', content_type='locution', relation='', input_text='The covid vaccine is safe', similar_illocutionary_force='Assertive Questioning', related_illocutionary_force='', similar_text='How is it justifiable to cut it', related_text='', example